In [1]:
import ast
import pandas as pd
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

vector_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "vector_retrieval_results.csv")
graph_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "graph_retrieval_results.csv")
hybrid_results = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "hybrid_retrieval_results.csv")

vector_results.head(), graph_results.head(), hybrid_results.head()

(  query_artist  rank retrieved_artist  similarity_score
 0  Soundgarden     1        Pearl Jam          0.740760
 1  Soundgarden     2          Nirvana          0.711962
 2  Soundgarden     3             Blur          0.684405
 3      Nirvana     1        Pearl Jam          0.754655
 4      Nirvana     2      Soundgarden          0.711962,
   query_artist  rank       retrieved_artist  graph_score evidence_type  \
 0         Blur     1              Pearl Jam            7    shared_tag   
 1         Blur     2  The Smashing Pumpkins            5    shared_tag   
 2         Blur     3                Nirvana            4    shared_tag   
 3     Deftones     1            Soundgarden            2    shared_tag   
 4     Deftones     2                Nirvana            2    shared_tag   
 
                                             evidence  
 0  ['alternative rock', 'grunge', 'rock', 'art ro...  
 1  ['alternative rock', 'grunge', 'rock', 'shoega...  
 2  ['alternative rock', 'grunge', 'n

In [3]:
vector_top1 = vector_results[vector_results["rank"] == 1][
    ["query_artist", "retrieved_artist", "similarity_score"]
].rename(columns={
    "retrieved_artist": "vector_top1",
    "similarity_score": "vector_score"
})

vector_top1

,query_artist,vector_top1,vector_score
0,Soundgarden,Pearl Jam,0.740760
3,Nirvana,Pearl Jam,0.754655
6,Deftones,Blur,0.691868
9,Pearl Jam,Blur,0.760920
12,The Smashing Pumpkins,Blur,0.696780
15,Blur,Pearl Jam,0.760920


In [4]:
hybrid_top1 = hybrid_results[hybrid_results["hybrid_rank"] == 1][
    [
        "query_artist",
        "retrieved_artist",
        "similarity_score",
        "shared_member",
        "shared_tag",
        "graph_bonus",
        "hybrid_score"
    ]
].rename(columns={
    "retrieved_artist": "hybrid_top1",
    "similarity_score": "hybrid_vector_score"
})

hybrid_top1

,query_artist,hybrid_top1,hybrid_vector_score,shared_member,shared_tag,graph_bonus,hybrid_score
0,Blur,Pearl Jam,0.760920,0.0,7.0,0.100000,0.860920
3,Deftones,Soundgarden,0.681798,0.0,2.0,0.100000,0.781798
6,Nirvana,Soundgarden,0.711962,1.0,6.0,0.285714,0.997676
9,Pearl Jam,Soundgarden,0.740760,1.0,4.0,0.257143,0.997903
12,Soundgarden,Nirvana,0.711962,1.0,6.0,0.300000,1.011961
15,The Smashing Pumpkins,Blur,0.696780,0.0,5.0,0.100000,0.796780


In [5]:
graph_ranked = graph_results.copy()

graph_ranked["evidence_priority"] = graph_ranked["evidence_type"].map({
    "shared_member": 2,
    "shared_tag": 1
}).fillna(0)

graph_top1 = graph_ranked.sort_values(
    ["query_artist", "evidence_priority", "graph_score"],
    ascending=[True, False, False]
).drop_duplicates("query_artist")

graph_top1 = graph_top1[
    ["query_artist", "retrieved_artist", "graph_score", "evidence_type", "evidence"]
].rename(columns={
    "retrieved_artist": "graph_top1",
    "graph_score": "graph_score",
    "evidence_type": "graph_evidence_type",
    "evidence": "graph_evidence"
})

graph_top1

,query_artist,graph_top1,graph_score,graph_evidence_type,graph_evidence
0,Blur,Pearl Jam,7,shared_tag,"['alternative rock', 'grunge', 'rock', 'art ro..."
3,Deftones,Soundgarden,2,shared_tag,"['alternative metal', 'alternative rock']"
6,Nirvana,Soundgarden,1,shared_member,['Jason Everman']
10,Pearl Jam,Soundgarden,1,shared_member,['Matt Cameron']
14,Soundgarden,Nirvana,1,shared_member,['Jason Everman']
19,The Smashing Pumpkins,Blur,5,shared_tag,"['alternative rock', 'grunge', 'rock', 'shoega..."


In [6]:
final_comparison = vector_top1.merge(
    graph_top1,
    on="query_artist",
    how="left"
).merge(
    hybrid_top1,
    on="query_artist",
    how="left"
)

final_comparison["top1_changed"] = (
    final_comparison["vector_top1"] != final_comparison["hybrid_top1"]
)

final_comparison

,query_artist,vector_top1,vector_score,graph_top1,graph_score,graph_evidence_type,graph_evidence,hybrid_top1,hybrid_vector_score,shared_member,shared_tag,graph_bonus,hybrid_score,top1_changed
0,Soundgarden,Pearl Jam,0.740760,Nirvana,1,shared_member,['Jason Everman'],Nirvana,0.711962,1.0,6.0,0.300000,1.011961,True
1,Nirvana,Pearl Jam,0.754655,Soundgarden,1,shared_member,['Jason Everman'],Soundgarden,0.711962,1.0,6.0,0.285714,0.997676,True
2,Deftones,Blur,0.691868,Soundgarden,2,shared_tag,"['alternative metal', 'alternative rock']",Soundgarden,0.681798,0.0,2.0,0.100000,0.781798,True
3,Pearl Jam,Blur,0.760920,Soundgarden,1,shared_member,['Matt Cameron'],Soundgarden,0.740760,1.0,4.0,0.257143,0.997903,True
4,The Smashing Pumpkins,Blur,0.696780,Blur,5,shared_tag,"['alternative rock', 'grunge', 'rock', 'shoega...",Blur,0.696780,0.0,5.0,0.100000,0.796780,False
5,Blur,Pearl Jam,0.760920,Pearl Jam,7,shared_tag,"['alternative rock', 'grunge', 'rock', 'art ro...",Pearl Jam,0.760920,0.0,7.0,0.100000,0.860920,False


In [7]:
graph_pair_scores = graph_results.pivot_table(
    index=["query_artist", "retrieved_artist"],
    columns="evidence_type",
    values="graph_score",
    aggfunc="sum",
    fill_value=0
).reset_index()

graph_pair_scores.columns.name = None

for col in ["shared_member", "shared_tag"]:
    if col not in graph_pair_scores.columns:
        graph_pair_scores[col] = 0

vector_top1_graph_evidence = graph_pair_scores.rename(columns={
    "retrieved_artist": "vector_top1",
    "shared_member": "vector_top1_shared_member",
    "shared_tag": "vector_top1_shared_tag"
})[
    [
        "query_artist",
        "vector_top1",
        "vector_top1_shared_member",
        "vector_top1_shared_tag"
    ]
]

final_comparison = final_comparison.merge(
    vector_top1_graph_evidence,
    on=["query_artist", "vector_top1"],
    how="left"
)

final_comparison[["vector_top1_shared_member", "vector_top1_shared_tag"]] = final_comparison[
    ["vector_top1_shared_member", "vector_top1_shared_tag"]
].fillna(0)

final_comparison

,query_artist,vector_top1,vector_score,graph_top1,graph_score,graph_evidence_type,graph_evidence,hybrid_top1,hybrid_vector_score,shared_member,shared_tag,graph_bonus,hybrid_score,top1_changed,vector_top1_shared_member,vector_top1_shared_tag
0,Soundgarden,Pearl Jam,0.740760,Nirvana,1,shared_member,['Jason Everman'],Nirvana,0.711962,1.0,6.0,0.300000,1.011961,True,1.0,4.0
1,Nirvana,Pearl Jam,0.754655,Soundgarden,1,shared_member,['Jason Everman'],Soundgarden,0.711962,1.0,6.0,0.285714,0.997676,True,0.0,7.0
2,Deftones,Blur,0.691868,Soundgarden,2,shared_tag,"['alternative metal', 'alternative rock']",Soundgarden,0.681798,0.0,2.0,0.100000,0.781798,True,0.0,0.0
3,Pearl Jam,Blur,0.760920,Soundgarden,1,shared_member,['Matt Cameron'],Soundgarden,0.740760,1.0,4.0,0.257143,0.997903,True,0.0,7.0
4,The Smashing Pumpkins,Blur,0.696780,Blur,5,shared_tag,"['alternative rock', 'grunge', 'rock', 'shoega...",Blur,0.696780,0.0,5.0,0.100000,0.796780,False,0.0,5.0
5,Blur,Pearl Jam,0.760920,Pearl Jam,7,shared_tag,"['alternative rock', 'grunge', 'rock', 'art ro...",Pearl Jam,0.760920,0.0,7.0,0.100000,0.860920,False,0.0,7.0


In [8]:
def interpret_final_result(row):
    if not row["top1_changed"]:
        return "Vector and hybrid agree on the top result."

    if row["shared_member"] > 0 and row["vector_top1_shared_member"] > 0:
        return "Hybrid changed the top result, but both vector and hybrid top results have shared-member graph evidence."

    if row["shared_member"] > 0:
        return "Hybrid changed the top result using shared-member graph evidence."

    if row["shared_tag"] > 0:
        return "Hybrid changed the top result using shared-tag graph evidence."

    return "Hybrid changed the top result, but the graph evidence should be checked."

In [9]:
final_comparison["finding"] = final_comparison.apply(interpret_final_result, axis=1)
final_comparison

,query_artist,vector_top1,vector_score,graph_top1,graph_score,graph_evidence_type,graph_evidence,hybrid_top1,hybrid_vector_score,shared_member,shared_tag,graph_bonus,hybrid_score,top1_changed,vector_top1_shared_member,vector_top1_shared_tag,finding
0,Soundgarden,Pearl Jam,0.740760,Nirvana,1,shared_member,['Jason Everman'],Nirvana,0.711962,1.0,6.0,0.300000,1.011961,True,1.0,4.0,"Hybrid changed the top result, but both vector..."
1,Nirvana,Pearl Jam,0.754655,Soundgarden,1,shared_member,['Jason Everman'],Soundgarden,0.711962,1.0,6.0,0.285714,0.997676,True,0.0,7.0,Hybrid changed the top result using shared-mem...
2,Deftones,Blur,0.691868,Soundgarden,2,shared_tag,"['alternative metal', 'alternative rock']",Soundgarden,0.681798,0.0,2.0,0.100000,0.781798,True,0.0,0.0,Hybrid changed the top result using shared-tag...
3,Pearl Jam,Blur,0.760920,Soundgarden,1,shared_member,['Matt Cameron'],Soundgarden,0.740760,1.0,4.0,0.257143,0.997903,True,0.0,7.0,Hybrid changed the top result using shared-mem...
4,The Smashing Pumpkins,Blur,0.696780,Blur,5,shared_tag,"['alternative rock', 'grunge', 'rock', 'shoega...",Blur,0.696780,0.0,5.0,0.100000,0.796780,False,0.0,5.0,Vector and hybrid agree on the top result.
5,Blur,Pearl Jam,0.760920,Pearl Jam,7,shared_tag,"['alternative rock', 'grunge', 'rock', 'art ro...",Pearl Jam,0.760920,0.0,7.0,0.100000,0.860920,False,0.0,7.0,Vector and hybrid agree on the top result.


In [10]:
total_queries = len(final_comparison)

changed_count = final_comparison["top1_changed"].sum()

shared_member_changes = (
    (final_comparison["top1_changed"]) &
    (final_comparison["shared_member"] > 0)
).sum()

shared_tag_changes = (
    (final_comparison["top1_changed"]) &
    (final_comparison["shared_member"] == 0) &
    (final_comparison["shared_tag"] > 0)
).sum()

summary = pd.DataFrame({
    "metric": [
        "Total query artists",
        "Hybrid changed top-1 result",
        "Changes driven by shared-member evidence",
        "Changes driven by shared-tag evidence",
        "Top-1 change rate"
    ],
    "value": [
        total_queries,
        changed_count,
        shared_member_changes,
        shared_tag_changes,
        f"{changed_count / total_queries:.1%}"
    ]
})

summary

,metric,value
0,Total query artists,6
1,Hybrid changed top-1 result,4
2,Changes driven by shared-member evidence,3
3,Changes driven by shared-tag evidence,1
4,Top-1 change rate,66.7%


In [11]:
out_path = PROJECT_ROOT / "data" / "processed" / "final_retrieval_comparison.csv"

final_comparison.to_csv(out_path, index=False)

out_path

WindowsPath('c:/uni/seriousism/reminisGraph/data/processed/final_retrieval_comparison.csv')

#### Phase 11: Final Retrieval Comparison

This notebook compares the final outputs from vector retrieval, graph retrieval, and hybrid retrieval.

The goal is to summarize what each retrieval method captures and identify when hybrid retrieval improves the top result by using graph-based evidence.

#### Phase 11 Conclusion

This notebook compares vector, graph, and hybrid retrieval results side by side.

The vector retriever captures text and tag-based similarity using embedding cosine similarity. The graph retriever captures explicit MusicBrainz relationship evidence, such as shared members and shared tags. The hybrid retriever combines both signals.

The final comparison shows when the hybrid retriever changes the top-1 result compared to the vector baseline. Changes driven by shared-member evidence are especially important because they represent historical or relational connections that vector similarity alone may not prioritize.

This evaluation supports the project’s core idea: vector retrieval is useful for style and metadata similarity, while graph retrieval adds relationship-based evidence. Hybrid retrieval can combine both signals to improve music discovery.

The output is saved as `final_retrieval_comparison.csv` and can be used for the final README, portfolio summary, and project findings.